In [20]:
import os
from hydra import initialize, initialize_config_module, initialize_config_dir, compose
from omegaconf import OmegaConf
from dotenv import load_dotenv
import trackio
import numpy as np
import gymnasium as gym
import torch
import lancedb
from tqdm.notebook import tqdm
import pyarrow as pa

In [21]:
with initialize(version_base=None, config_path="conf"):
    load_dotenv()
    write_url = os.environ["TRACKIO_WRITE_URL"]
    cfg = compose(config_name="config", overrides=[f'trackio.write_url="{write_url}"'])
    db = lancedb.connect(cfg.dataset.lancedb_uri)
    image_inner_shape = [96, 96, 3]
    action_inner_shape = [3]
    
    image_type = pa.fixed_shape_tensor(pa.uint8(), shape=image_inner_shape)
    action_type = pa.fixed_shape_tensor(pa.float32(), shape=action_inner_shape)
    
    schema = pa.schema([
        pa.field("episode_id", pa.uint32()),
        pa.field("observations", pa.list_(pa.list_(pa.uint8(), 96*96*3))), 
        pa.field("actions", pa.list_(pa.list_(pa.float32(), 3)))
    ])
    
    # Create the table once (using an empty table or your first batch)
    tbl = db.create_table("episodes",pa.Table.from_batches([], schema=schema), schema=schema, mode="overwrite")

In [22]:
envs = gym.make_vec("CarRacing-v3", render_mode="rgb_array", lap_complete_percent=0.95, domain_randomize=False, continuous=True, num_envs=os.cpu_count(), vectorization_mode="async")

In [23]:
ids = np.array([i for i in range(os.cpu_count())])

for _ in tqdm(range(cfg.dataset.episodes//os.cpu_count())):
    obs, _ = envs.reset()
    observations = [obs]
    actions = []
    endings = [np.array(False)]

    while not np.all(endings[-1]):
        action = envs.action_space.sample()
        observation, _, terminated, truncated, _ = envs.step(action)
        observations.append(observation)
        actions.append(action)
        episode_over = np.logical_or(terminated, truncated)
        endings.append(episode_over)
    del endings[0]

    endings_ = np.stack(endings, axis=1)
    observations_ = np.stack(observations, axis=1)
    actions_ = np.stack(actions, axis=1)
    ends = [len(endings_[i]) - 1 - endings_[i][::-1].argmin() for i in range(os.cpu_count())]

    row_lengths = [ends[i].item() + 1 for i in range(os.cpu_count())]
    offsets = [0] + np.cumsum(row_lengths).tolist()

    # Observations: list<fixed_size_list<uint8>[27648]>
    obs_chunks = [observations_[i][0:row_lengths[i]] for i in range(os.cpu_count())]
    flat_obs_numpy = np.concatenate(obs_chunks, axis=0)
    flat_obs_tensor = pa.FixedSizeListArray.from_arrays(
        pa.array(flat_obs_numpy.reshape(-1), type=pa.uint8()), 96 * 96 * 3
    )
    total_obs = pa.ListArray.from_arrays(offsets, flat_obs_tensor)

    # Actions: list<fixed_size_list<float32>[3]>
    act_chunks = [actions_[i][0:row_lengths[i]] for i in range(os.cpu_count())]
    flat_act_numpy = np.concatenate(act_chunks, axis=0).astype(np.float32)
    flat_act_tensor = pa.FixedSizeListArray.from_arrays(
        pa.array(flat_act_numpy.reshape(-1), type=pa.float32()), 3
    )
    total_acts = pa.ListArray.from_arrays(offsets, flat_act_tensor)

    data = pa.Table.from_arrays(
        [ids, total_obs, total_acts],
        schema=schema
    )

    tbl.add(data)
    ids = ids + os.cpu_count()

  0%|          | 0/12 [00:00<?, ?it/s]